# Neural Networks III: Improving Generalization

Until now, we have focused on how to build and train neural network models that fit the training data.

We have also seen that neural networks are incredibly powerful — capable of fitting **any function**. However, this flexibility comes with a downside: they are prone to **overfitting**, especially as we increase the number of hidden layers or model complexity.

In this lesson, we will introduce several key techniques for improving **generalization**, often by mitigating overfitting, in neural networks.

By the end of this lesson, you should be able to understand and implement these techniques in `PyTorch`, including:

- Early stopping
- Dropout
- L2 Regularization (Weight Decay)
- Batch normalization

## 1. Early Stopping

If you let neural networks train for too long, their ability to generalize to unseen data may start to **deteriorate** — a classic symptom of **overfitting**.

One way to prevent this is to **stop the training process as soon as performance on a tuning set begins to degrade**. This is the core idea behind the **early-stopping** criterion.

Early stopping is a widely used and effective regularization technique.

In [ ]:
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pylab as plt
import numpy as np

# This is the data to use for this tutorial

X, y = load_digits()["data"], load_digits()["target"]


# Let's visualize a few digits
fig, axis = plt.subplots(figsize=(5,7), ncols=2, nrows=5)
axis = axis.flatten()
for i_digit in np.unique(y):
    axis[i_digit].imshow(X[y==i_digit][0,:].reshape(-8, 8))
plt.tight_layout()

In [ ]:
# This is a multiclass problem with a softmax output layer: we need to one-hot-encode our classes
y = OneHotEncoder(sparse_output=False).fit_transform(y.reshape(-1,1))

X = torch.tensor(X / 16.0, dtype=torch.float32)
y = torch.tensor(y.astype(int), dtype=torch.long)

X_fit, X_tune, y_fit, y_tune = train_test_split(X, y, random_state=0, test_size=100, train_size=400, stratify=y)

# Define batch size
batch_size = 64

# HERE COMES CODE: Wrap the fitting data in a TensorDataset and DataLoader
fit_dataset = None
fit_loader = None

# HERE COMES CODE: Wrap the tuning data in a TensorDataset and DataLoader
tune_dataset = None
tune_loader = None

In [ ]:
from torch import nn

# Let's create this simple neural network (100 -> 100--> 10)
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_1 = None # HERE COMES CODE: Training -> 100
        self.linear_2 = None # HERE COMES CODE: 100 -> 100
        self.linear_3 = None # HERE COMES CODE: 100 -> 10
        self.relu = None # HERE COMES CODE: ReLU Activation
        self.softmax = None # HERE COMES CODE: Softamx Activation (set dim=1, be careful with this!)

    def forward(self, x):
        # CODE COMES HERE: Peform passing linear_1 -> Relu -> linear_2 -> ReLU --> linear_3 --> softmax
        return None

---
💡 **Quick Exercise**

Accomplish the same class definition, including its `__init__` and `forward` methods with `nn.Modulelist`


In [ ]:
# YOUR CODE

---

In [ ]:
# The implementation of Categorical Cross Entropy (see previous lab!)
class CategoricalCrossEntropy(nn.Module):
    def forward(self, y_pred, y_true):
        cost = 0
        for ii in range(y_true.shape[1]):
            y_oc = y_true[:, ii]
            y_pc = y_pred[:, ii]
        
            cost -= y_oc*torch.log(y_pc.clamp_min(1e-12))

        return cost.mean(0) # This is just to average across all observations    

In [ ]:
from torch import optim

torch.manual_seed(123)

model = None # HERE COMES CODE: Instantiate model
cost_fn = None # HERE COMES CODE: use the categorical cross entropy cost function above! 
optimizer = None # HERE COMES CODE: Adam

# Train and track losses
n_epochs = 500
fit_costs = []
tune_costs = []

for epoch in range(n_epochs):
    model.train()
    running_cost = 0
    for X_batch, y_batch in fit_loader:
        y_pred = None # CODE COMES HERE: Calculate output
        fit_cost = None # HERE COMES CODE: calculate cost
        # CODE COMES HERE: Calculate gradients
        # CODE COMES HERE: Apply gradients
        # HERE COMES CODE: Gradients equal to zero
        running_cost += 0.0 # Replace: fit_cost.item() * X_batch.shape[0]

    fit_costs.append(running_cost / X_fit.shape[0])

    # Let's keep track of the lost in the tuning set:
    model.eval()
    running_cost = 0
    with torch.no_grad():
        for X_batch, y_batch in tune_loader:
            y_pred = None # HERE COMES CODE: Calculate output
            tune_cost = None # HERE COMES CODE: calculate cost
            running_cost +=  0.0 # Replace: tune_cost.item() * X_batch.shape[0]

    tune_costs.append(running_cost/ X_tune.shape[0])

Let's calculate the performance of our model on the fitting and tuning sets.

Remember, our output layer contains 10 units, one for each class. 

In [ ]:
model.eval()
with torch.no_grad():
    print(model(X_tune[[0]]))

Remember that softmax in the output layer returns probabilities for each class. Therefore, we just need to take that unit with the highest value as our output label, since this will be our most likely class.

In [ ]:
with torch.no_grad():
    x = X_tune[:2]
    probs = model(x)
    print("probs: ", probs.numpy())
    print("these probs should sum 1:", probs.sum(dim=1))
    print(torch.argmax(probs, dim=1))

And obviously, when comparing with the observed values, we will need to also take the unit with the highest value, because our data is one-hot-encoded:

In [ ]:
print(y_tune[:2, :])
print(torch.argmax(y_tune[:2, :], dim=1))

In [ ]:
from sklearn.metrics import accuracy_score

model.eval() 
with torch.no_grad():  # No gradient tracking

    y_fit_pred = torch.argmax(model(X_fit), dim=1)
    y_tune_pred = torch.argmax(model(X_tune), dim=1)

acc_fit = accuracy_score(torch.argmax(y_fit, axis=1), y_fit_pred)
acc_tune = accuracy_score(torch.argmax(y_tune, axis=1), y_tune_pred)

In [ ]:
import matplotlib.pylab as plt

# 6. Plot the fitting and tuning cost
plt.plot(fit_costs, label=f"Fit (ACC={acc_fit:.3f})")
plt.plot(tune_costs, label=f"Tuning (ACC={acc_tune:.3f})")
plt.xlabel("Epoch")
plt.ylabel("Cost")
plt.title("Learning Curves")
plt.legend()
plt.grid(True)
plt.show()

As you can see, after a few epochs, our tuning loss starts to increase. Ideally, we would stop training at that point.

### Implementation

`PyTorch` does not include a built-in implementation of early stopping, but it is not difficult to implement. Below is one possible recipe generated by an LLM:

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, tune_loss, model):
        if self.best_loss is None or tune_loss < self.best_loss - self.min_delta:
            self.best_loss = tune_loss
            self.counter = 0
            self.best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def restore_best_weights(self, model):
        if self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)

This may seem complicated. Other libraries, such as `scikit-learn` and `Keras`, provide early-stopping implementations.

However, implementing it yourself helps you understand how it works internally and build useful intuition.

In [ ]:
early_stopping = EarlyStopping(patience=5)

torch.manual_seed(123)

model = MLP()
cost_fn = CategoricalCrossEntropy()
optimizer = optim.Adam(model.parameters())

# Train and track losses
n_epochs = 500
fit_costs = []
tune_costs = []

for epoch in range(n_epochs):
    model.train()
    running_cost = 0
    for X_batch, y_batch in fit_loader:
        y_pred = model(X_batch)
        fit_cost = cost_fn(y_pred, y_batch)
        fit_cost.backward()
        optimizer.step()
        optimizer.zero_grad()
        running_cost += fit_cost.item() * X_batch.shape[0]

    fit_costs.append(running_cost / X_fit.shape[0])

    model.eval()
    running_cost = 0
    with torch.no_grad():
        for X_batch, y_batch in tune_loader:
            y_pred = model(X_batch)
            tune_cost = cost_fn(y_pred, y_batch)
            running_cost += tune_cost.item() * X_batch.shape[0]

    tune_costs.append(running_cost/ X_tune.shape[0])

    #--- Early Stopping --- 
    early_stopping(running_cost / X_tune.shape[0], model)

    if early_stopping.early_stop:
        print(f"Stopping early at epoch {epoch+1}")
        break

# This restores the best model
early_stopping.restore_best_weights(model)

#--- Early Stopping ---

In [ ]:
model.eval()
with torch.no_grad():  # No gradient tracking

    y_fit_pred = torch.argmax(model(X_fit), dim=1)
    y_tune_pred = torch.argmax(model(X_tune), dim=1)

acc_fit = accuracy_score(np.argmax(y_fit, axis=1), y_fit_pred)
acc_tune = accuracy_score(np.argmax(y_tune, axis=1), y_tune_pred)

In [ ]:
# 6. Plot the fitting and tuning loss
plt.plot(fit_costs, label=f"Fit (ACC={acc_fit:.3f})")
plt.plot(tune_costs, label=f"Tuning (ACC={acc_tune:.3f})")
plt.xlabel("Epoch")
plt.ylabel("Cost")
plt.title("Learning Curves")
plt.legend()
plt.grid(True)
plt.show()

## 2. Dropout

One effective technique to prevent overfitting is randomly deactivating neurons (units) during training. This is the main idea behind **dropout**.

Dropout forces the model to not rely too heavily on any single neuron; instead, it encourages the network to learn more robust and distributed patterns.

It is typically applied to the hidden layers of the network. You can apply dropout to one layer or to multiple layers, depending on the model's complexity and your regularization needs.

This technique is so effective that most modern neural networks include some form of dropout by default.

### Implementation

In `PyTorch`, dropout is implemented as a type of layer because it acts as a transformation applied to layer activations. Specifically, it randomly sets a subset of the units to zero during training.

We will typically use `nn.Dropout` for this. The proportion of units to be, randomly, set to zero is controlled by the attribute `p` in this class. Check out its [documentation](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Dropout.html#torch.nn.Dropout).

You can find other types in the [dropout layers documentation](https://docs.pytorch.org/docs/stable/nn.html#dropout-layers).

<div class="alert alert-block alert-info">

**NOTE**  
Dropout is active **only during training**. When calling `model.eval()`, dropout is automatically disabled.

That is why we use `model.eval()` when evaluating performance on the tuning set. The tuning set guides model selection; reserve a separate reporting set for final evaluation. This notebook does not create a separate reporting set.

</div>


In [ ]:
class MLPwDropOut(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_1 = nn.Linear(X_fit.shape[1], 100)
        self.linear_2 = nn.Linear(100, 100)
        self.linear_3 = nn.Linear(100, 10)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1) # Softamx Activation (here it needs dim=1, be careful with this!)
        self.dropout = None # HERE COMES CODE dropout with p=0.2
        
    def forward(self, x):
        x = self.relu(self.linear_1(x))
        x = None # HERE COMES CODE: Apply droout to the first hidden layer
        x = self.relu(self.linear_2(x))
        x = None # HERE COMES CODE: Apply droout to the second hidden layer
        x = self.softmax(self.linear_3(x))
        return x

In [ ]:
early_stopping = EarlyStopping(patience=5)

torch.manual_seed(123)

model = MLPwDropOut()
cost_fn = CategoricalCrossEntropy()
optimizer = optim.Adam(model.parameters())

# Train and track losses
n_epochs = 500
fit_costs = []
tune_costs = []

for epoch in range(n_epochs):
    model.train()
    running_cost = 0
    for X_batch, y_batch in fit_loader:
        y_pred = model(X_batch)
        fit_cost = cost_fn(y_pred, y_batch)
        fit_cost.backward()
        optimizer.step()
        optimizer.zero_grad()
        running_cost += fit_cost.item() * X_batch.shape[0]

    fit_costs.append(running_cost / X_fit.shape[0])

    model.eval()
    running_cost = 0
    with torch.no_grad():
        for X_batch, y_batch in tune_loader:
            y_tune_pred = model(X_batch)
            tune_cost = cost_fn(y_tune_pred, y_batch)
            running_cost += tune_cost.item() * X_batch.shape[0]

    tune_costs.append(running_cost/ X_tune.shape[0])
    
    early_stopping(running_cost / X_tune.shape[0], model)

    if early_stopping.early_stop:
        print(f"Stopping early at epoch {epoch+1}")
        break

# This restores the best model
early_stopping.restore_best_weights(model)

In [ ]:
model.eval()
with torch.no_grad():  # No gradient tracking

    y_fit_pred = torch.argmax(model(X_fit), dim=1)
    y_tune_pred = torch.argmax(model(X_tune), dim=1)

acc_fit = accuracy_score(np.argmax(y_fit, axis=1), y_fit_pred)
acc_tune = accuracy_score(np.argmax(y_tune, axis=1), y_tune_pred)

In [ ]:
# 6. Plot the fitting and tuning loss
plt.plot(fit_costs, label=f"Fit (ACC={acc_fit:.3f})")
plt.plot(tune_costs, label=f"Tuning (ACC={acc_tune:.3f})")
plt.xlabel("Epoch")
plt.ylabel("Cost")
plt.title("Learning Curves")
plt.legend()
plt.grid(True)
plt.show()

As you can see, the separation between the fitting and tuning cost has decreased a bit. In this case it means the model is not just memorizing the training data, but learning patterns that generalize better. Dropout helped reduce overfitting.

> **Question:** Why didn't we apply dropout to the last, output layer, i.e., right after applying the softmax activation?


## 3. L2 Regularization (Weight Decay)

L2 regularization adds a penalty to the loss function for large weights. This can help prevent overfitting and improve the model's generalization. This approach is also known as **weight decay**.

### Implementation

In `PyTorch`, this regularization is commonly applied through the optimizer. The idea is the same: penalize large weights to help prevent overfitting.


> optimizer = optim.Adam(model.parameters(), **weight_decay=1e-2**)


In other libraries such as `Keras`, L2 penalties are typically attached directly to individual layers. For more details, you can refer to [Keras's documentation](https://keras.io/api/layers/regularizers).

In [ ]:
early_stopping = EarlyStopping(patience=5)

torch.manual_seed(123)

model = MLPwDropOut()
cost_fn = CategoricalCrossEntropy()
optimizer = None # HERE COMES CODE: Define adam with weight_decay

# Train and track losses
n_epochs = 500
fit_costs = []
tune_costs = []

for epoch in range(n_epochs):
    model.train()
    running_cost = 0
    for X_batch, y_batch in fit_loader:
        y_pred = model(X_batch)
        fit_cost = cost_fn(y_pred, y_batch)
        fit_cost.backward()
        optimizer.step()
        optimizer.zero_grad()
        running_cost += fit_cost.item() * X_batch.shape[0]

    fit_costs.append(running_cost / X_fit.shape[0])

    model.eval()
    running_cost = 0
    with torch.no_grad():
        for X_batch, y_batch in tune_loader:
            y_tune_pred = model(X_batch)
            tune_cost = cost_fn(y_tune_pred, y_batch)
            running_cost += tune_cost.item() * X_batch.shape[0]

    tune_costs.append(running_cost/ X_tune.shape[0])
    
    early_stopping(running_cost / X_tune.shape[0], model)

    if early_stopping.early_stop:
        print(f"Stopping early at epoch {epoch+1}")
        break

# This restores the best model
early_stopping.restore_best_weights(model)

In [ ]:
model.eval()
with torch.no_grad():  # No gradient tracking

    y_fit_pred = torch.argmax(model(X_fit), dim=1)
    y_tune_pred = torch.argmax(model(X_tune), dim=1)

acc_fit = accuracy_score(np.argmax(y_fit, axis=1), y_fit_pred)
acc_tune = accuracy_score(np.argmax(y_tune, axis=1), y_tune_pred)

In [ ]:
# 6. Plot the fitting and tuning loss
plt.plot(fit_costs, label=f"Fit (ACC={acc_fit:.3f})")
plt.plot(tune_costs, label=f"Tuning (ACC={acc_tune:.3f})")
plt.xlabel("Epoch")
plt.ylabel("Cost")
plt.title("Learning Curves")
plt.legend()
plt.grid(True)
plt.show()

---
💡 **Quick Exercise**

With regularization, more is not always better.

Try setting `weight_decay=1e-4` in the optimizer. What do you observe in terms of fitting and tuning performance? Why do you think this happens?


In [ ]:
# YOUR CODE

---

## 4. Batch Normalization

Batch normalization standardizes the inputs to each layer by subtracting the batch mean and dividing by the batch standard deviation.

**This helps stabilize and accelerate training, and often acts as a form of mild regularization.**

### Implementation

In `PyTorch`, you can just insert `nn.BatchNorm1d(n_units)` after a linear layer and before the activation:

>
    x = self.linear_1(x)
    x = self.bn1(x)
    x = self.relu(x)


In [ ]:
class MLPwDropOutAndBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_1 = nn.Linear(X_fit.shape[1], 100)
        self.linear_2 = nn.Linear(100, 100)
        self.linear_3 = nn.Linear(100, 10)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1) # Softamx Activation (here it needs dim=1, be careful with this!)
        self.dropout = nn.Dropout(p=0.2) 
        self.bn1 = None # HERE COMES CODE: Define BatchNormalization with 100
        self.bn2 = None # HERE COMES CODE: Define BatchNormalization with 100
        
    def forward(self, x):
        x = self.linear_1(x)
        x = None # HERE COMES CODE: # Apply the first batch normalization after the first linear transformation
        x = self.relu(x)
        x = self.dropout(x) 
        x = self.linear_2(x)
        x = None # HERE COMES CODE: # Apply the second batch normalization after the second linear transformation
        x = self.relu(x)
        x = self.dropout(x) 
        x = self.linear_3(x)
        x = self.softmax(x)
        
        return x

<div class="alert alert-block alert-warning">

🚨⚠️ **CAUTION** 🚨⚠️

You must use separate `BatchNorm1d` layers, even if all linear layers have the same number of units, as in our example.

`BatchNorm1d` computes statistics (mean and standard deviation) from the activations of each specific layer in the network. If you reuse the same `BatchNorm1d` instance, you would be applying the **same normalization statistics across multiple layers**, which is incorrect.

Think of each `BatchNorm1d` as being tied to the specific context of the layer it follows — not just the shape. Just like you would not reuse the same Linear(100, 100) layer twice, even if it has the same shape, you should not reuse the same BatchNorm layer either.

</div>

In [ ]:
early_stopping = EarlyStopping(patience=5)

torch.manual_seed(123)

model = MLPwDropOutAndBatchNorm()
cost_fn = CategoricalCrossEntropy()
optimizer = optim.Adam(model.parameters(), weight_decay=1e-2)

# Train and track losses
n_epochs = 500
fit_costs = []
tune_costs = []

for epoch in range(n_epochs):
    model.train()
    running_cost = 0
    for X_batch, y_batch in fit_loader:
        y_pred = model(X_batch)
        fit_cost = cost_fn(y_pred, y_batch)
        fit_cost.backward()
        optimizer.step()
        optimizer.zero_grad()
        running_cost += fit_cost.item() * X_batch.shape[0]

    fit_costs.append(running_cost / X_fit.shape[0])

    model.eval()
    running_cost = 0
    with torch.no_grad():
        for X_batch, y_batch in tune_loader:
            y_pred = model(X_batch)
            tune_cost = cost_fn(y_pred, y_batch)
            running_cost += tune_cost.item() * X_batch.shape[0]

    tune_costs.append(running_cost/ X_tune.shape[0])
    
    early_stopping(running_cost / X_tune.shape[0], model)

    if early_stopping.early_stop:
        print(f"Stopping early at epoch {epoch+1}")
        break

early_stopping.restore_best_weights(model)

In [ ]:
model.eval()
with torch.no_grad():  # No gradient tracking

    y_fit_pred = torch.argmax(model(X_fit), dim=1)
    y_tune_pred = torch.argmax(model(X_tune), dim=1)

acc_fit = accuracy_score(np.argmax(y_fit, axis=1), y_fit_pred)
acc_tune = accuracy_score(np.argmax(y_tune, axis=1), y_tune_pred)

In [ ]:
# 6. Plot the fitting and tuning loss
plt.plot(fit_costs, label=f"Fit (ACC={acc_fit:.3f})")
plt.plot(tune_costs, label=f"Tuning (ACC={acc_tune:.3f})")
plt.xlabel("Epoch")
plt.ylabel("Cost")
plt.title("Learning Curves")
plt.legend()
plt.grid(True)
plt.show()

As you can see, batch normalization helped our model fit the training data perfectly. But it also seems to have reduced generalization a bit. Why?

Because, while it stabilizes training and speeds up learning, it does not directly prevent overfitting — and in some cases, it can overpower other forms of regularization like dropout or L2.

This is a great reminder that these tools are **not silver bullets**. They need to be **balanced and tuned together**.

---

💡 **Quick Exercise**

After adding `BatchNorm1d` layers to the model, you may notice that the fitting accuracy reaches 100%, while tuning accuracy decreases slightly.

Try adjusting:

- The **dropout rate** (e.g., increase from `0.2` to `0.5`, or reduce to `0.1`)
- The **L2 penalty** via `weight_decay` (e.g., try `1e-4` instead of `1e-5`)

Can you find settings that restore or improve tuning performance without sacrificing too much fitting accuracy?

Why do you think BatchNorm makes the model overfit more easily, even though it helps with convergence?


In [ ]:
# YOUR CODE

---

## 5. Summary and Looking Ahead: Deep Learning

In this module, we have learned how to build and train simple neural networks.

In this final lesson, we focused on helping those models generalize by using:

- **Early stopping** to avoid overtraining.
- **Dropout** to reduce overfitting.
- **L2 regularization** to constrain weights.
- **Batch Normalization** to stabilize and accelerate learning.

These techniques are not only useful for small, shallow networks. They are also **essential building blocks** for training **deep neural networks**, which can have dozens or even hundreds of layers.

For example:

- Batch Normalization was a key breakthrough that made deep learning practical.
- Dropout is still used in large-scale models such as transformers.
- Weight decay remains a core part of most modern optimizer setups.

If you choose the analytics concentration in our program, you will explore these ideas in greater depth in future courses.

In fact, many of the challenges in deep learning are simply **extensions of what you have already seen**: making models that are expressive, efficient, and able to generalize to new data.

You have also learned these concepts using `PyTorch`, which has helped you build intuition about how neural networks work and prepare for the programming in more advanced deep learning courses.

## 📝 6. Exercises

### 6.1 Exercise — Replicate on a regression task

You will now apply the same techniques to a regression task: predicting housing prices from features such as location, income, and population.

Given the dataset below, you should:

- Convert the fitting and tuning data to `DataLoader` objects to facilitate training.
- Build a fully connected neural network with **at least two hidden layers**.
- Try adding **dropout**, **L2 regularization**, and **batch normalization**.
- Try using **early stopping** to prevent overfitting.
- Compare fitting and tuning performance using an appropriate metric.
- Plot the predicted vs. true values for the tuning set.

What happens when you remove dropout or batch normalization? How does tuning `weight_decay` affect performance?

**Remember:** This is a regression problem. That affects the output layer, the choice of loss function, and the performance metrics.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(return_X_y=True)
X_fit, X_tune, y_fit, y_tune = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# YOUR CODE